## Notebook45

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install igraph --quiet

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSNetwork

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
case = pl.read_csv(ub + "data/scotus_case.csv")
cite = pl.read_csv(ub + "data/scotus_citation.csv")

### Questions

This notebook works with a citation network from the U.S. Supreme Court. Each row
of `case` is one decided case, identified by its `citation` (for example
`"384 U.S. 436"` is *Miranda v. Arizona*), with columns such as `case_name`, `term`
(the year), and `chief` (the chief justice). Each row of `cite` is one link: the
case in `citing_case` cites the earlier case in `cited_case`.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Using `case`, count the number of cases decided under each `chief` justice.
Sort from most to fewest cases.

In [ ]:
(
    case
    .group_by(c.chief)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
)

2. Filter `cite` to the rows where `citing_case` is `"384 U.S. 436"`
(*Miranda v. Arizona*). These are the cases that *Miranda* cites.

In [ ]:
(
    cite
    .filter(c.citing_case == "384 U.S. 436")
)

*Miranda* cites 81 earlier cases.

3. Count the number of times each case is cited by grouping `cite` by
`cited_case`. Sort from most to least cited and keep the first 10 rows.

In [ ]:
(
    cite
    .group_by(c.cited_case)
    .agg(n_in = pl.len())
    .sort(c.n_in, descending=True)
    .head(10)
)

4. Take the code from the previous question and, before sorting, join it to
`case.select(c.citation, c.case_name, c.term)` using `left_on="cited_case"` and
`right_on="citation"`, so that we can see the names of the cases.

In [ ]:
(
    cite
    .group_by(c.cited_case)
    .agg(n_in = pl.len())
    .join(
        case.select(c.citation, c.case_name, c.term),
        left_on="cited_case",
        right_on="citation"
    )
    .sort(c.n_in, descending=True)
    .head(10)
)

The most-cited case is *McCulloch v. Maryland* (1819), with 255 citations,
followed by *Miranda* and *Boyd v. United States*.

5. The full network is too big to draw, so the block below is given. It keeps the 80
most-cited cases and the citations among them, builds an undirected network with
`DSNetwork.process()`, and joins the case information onto the node table. The
results are saved as `node` and `edge`. Run it and look at `node`.

In [ ]:
top = (
    cite
    .group_by(c.cited_case)
    .agg(n_in = pl.len())
    .sort([c.n_in, c.cited_case], descending=[True, False])
    .head(80)
    .get_column("cited_case")
    .to_list()
)

edges = (
    cite
    .filter(c.citing_case.is_in(top), c.cited_case.is_in(top))
    .rename({"citing_case": "doc_id", "cited_case": "doc_id2"})
)

node, edge, G = DSNetwork.process(edges, directed=False)

node = (
    node
    .join(
        case.select(c.citation, c.case_name, c.term),
        left_on="id",
        right_on="citation",
        how="left"
    )
)
node

6. Draw the network, as in the chapter. Start with `node`, add the edges with
`.geom_segment(aes(x=c.x, y=c.y, xend=c.xend, yend=c.yend), data=edge, alpha=0.1)`,
then add the nodes with `.geom_point()`. Finish with `.theme_void()`.

In [ ]:
(
    node
    .ggplot(aes(c.x, c.y))
    .geom_segment(
        aes(x=c.x, y=c.y, xend=c.xend, yend=c.yend),
        data=edge,
        alpha=0.1
    )
    .geom_point()
    .theme_void()
)

The network has 77 nodes, not 80. Three of the most-cited cases do not cite, and
are not cited by, any of the other cases in the set, so they drop out.

7. Sort `node` by `degree` from largest to smallest, keep the first 10 rows, and
select `case_name`, `term`, and `degree`.

In [ ]:
(
    node
    .sort(c.degree, descending=True)
    .head(10)
    .select(c.case_name, c.term, c.degree)
)

The case with the highest degree is *Palko v. Connecticut*, with 23 links,
followed by *Gideon v. Wainwright*. *McCulloch*, the most-cited case overall, is not
in the top 10, because most of the cases that cite it are not in the network.

8. Take the code from question 6 and color the points by `degree`. Use
`.geom_point(aes(color=c.degree), size=3)`.

In [ ]:
(
    node
    .ggplot(aes(c.x, c.y))
    .geom_segment(
        aes(x=c.x, y=c.y, xend=c.xend, yend=c.yend),
        data=edge,
        alpha=0.1
    )
    .geom_point(aes(color=c.degree), size=3)
    .theme_void()
)

9. Do the same as question 7, but sort by the eigenvector centrality `eigen`. Select
`case_name`, `term`, `eigen`, and `degree`.

In [ ]:
(
    node
    .sort(c.eigen, descending=True)
    .head(10)
    .select(c.case_name, c.term, c.eigen, c.degree)
)

The top of the list is almost the same as for degree. Most of these are
criminal-procedure and free-speech cases from the twentieth century.

10. Do the same as question 7, but sort by the betweenness centrality `between`.
Select `case_name`, `between`, `degree`, and `eigen`.

In [ ]:
(
    node
    .sort(c.between, descending=True)
    .head(10)
    .select(c.case_name, c.between, c.degree, c.eigen)
)

*Ashwander v. TVA* and *McCulloch* have a degree of only 8 and a low `eigen`
score, but they are in the top 6 for betweenness. They are bridges between
different parts of the network.

11. Group `node` by `cluster` and list the members of each cluster with
`members = c.case_name.str.join("; ")` inside of `.agg`.

In [ ]:
(
    node
    .group_by(c.cluster)
    .agg(members = c.case_name.str.join("; "))
)

The clusters line up with areas of law. One cluster is all search-and-seizure
cases (*Boyd*, *Weeks*, *Katz*, *Terry*), and two antitrust cases (*Standard Oil*
and *Socony*) form their own cluster. One cluster has no name because its only
case, `"75 U.S. 168"`, has no row in `case`.